# WP2 - Pooled Building-Token Transformer (base vs Fourier/FiLM)

**Question.** Move from a raster CNN (WP1) to an object-based model: encode
buildings as tokens, pool them to one geometry vector `z_geom`, and decode the
velocity at any query coordinate. Two decoders isolate *where* the bottleneck is.

**Headline.**

| model | RMSE | MAE | R2 |
|---|---|---|---|
| WP1 - U-Net (raster) | 0.8217 | 0.4821 | 0.7194 |
| WP2 - Pooled (base) | 1.2796 | 0.8635 | 0.3195 |
| WP2 - Pooled + Fourier/FiLM | 1.1587 | 0.7722 | **0.4421** |

**Reading.** The base pooled model collapses toward a near-mean field. Fourier
features + FiLM recover a large fraction of that (an earlier base run sat at
R2 ~ 0.06, so 0.06 -> 0.44), which proves the decoder's spectral bias was *a*
bottleneck. It still does not reach the U-Net. The residual gap is mean-pooling:
one latent vector cannot carry per-location geometry. That is what makes WP3's
per-query cross-attention mandatory rather than optional.

> **Split note.** These numbers are on the original WP0 split (3,657 train /
> 785 test), matching the WP1-WP4 convention. WP5's controlled 4-model retrain
> uses a smaller reduced-core split (2,518 train / 541 test) so all models see
> identical exposure; the WP2-pool row there reads R2 = 0.2921. The two are
> different training-set sizes, not the same number - do not merge them.

Both models share the encoder (`Linear(5 -> d_model)` -> 3-layer Pre-LN
Transformer -> masked mean-pool -> `z_geom`) and expose the same
`encode / decode / forward` interface, so one training and evaluation path serves
both. The models and the token dataset live in the `urbanformer` package.

## 1. Setup

In [ ]:
import copy
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from urbanformer.data import TokenDataset, collate_fn
from urbanformer.models.pooled import PooledTransformer, PooledTransformerFiLM
from urbanformer.metrics import field_metrics, per_case_rmse

# --- paths (repo-relative) ---
DATA_ROOT  = Path("data/processed")
SPLITS_DIR = Path("data/splits")

# --- hyperparameters ---
K_PTS      = 2000    # fluid query points sampled per case per step
BATCH_SIZE = 32
NUM_EPOCHS = 50
LR         = 1e-3
PATIENCE   = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WP1_RMSE, WP1_MAE, WP1_R2 = 0.8217, 0.4821, 0.7194   # WP1 reference row
print("Device:", DEVICE)

## 2. Data

`TokenDataset` returns `(tokens [N_b, 5], query_xy [K_pts, 2], target [K_pts])`.
In `'train'` mode it samples `K_pts` random fluid points and shuffles token order
(buildings are a set, not a sequence). `collate_fn` pads variable-length token
sequences to the batch max and builds the boolean `padding_mask`
(`True = padding`, PyTorch's `src_key_padding_mask` convention).

In [ ]:
def load_split(split_file, data_root):
    """Read a split .txt of case names into a list of case directories."""
    with open(split_file) as f:
        names = [line.strip() for line in f if line.strip()]
    return [data_root / name for name in names]

train_dirs = load_split(SPLITS_DIR / "train_cases.txt", DATA_ROOT)
val_dirs   = load_split(SPLITS_DIR / "val_cases.txt",   DATA_ROOT)
test_dirs  = load_split(SPLITS_DIR / "test_cases.txt",  DATA_ROOT)
print(f"train/val/test: {len(train_dirs)} / {len(val_dirs)} / {len(test_dirs)}")

train_ds = TokenDataset(train_dirs, mode="train", K_pts=K_PTS)
val_ds   = TokenDataset(val_dirs,   mode="train", K_pts=K_PTS)   # same sampling for val

pin = (DEVICE.type == "cuda")
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn, pin_memory=pin)

tok_b, pmask_b, qxy_b, tgt_b = next(iter(train_loader))
print("tokens_pad", tuple(tok_b.shape), "| padding_mask", tuple(pmask_b.shape),
      "| query_xy", tuple(qxy_b.shape), "| target", tuple(tgt_b.shape))

## 3. Models

Both models share the token encoder and expose `encode -> decode -> forward`.
The base decoder concatenates `z_geom` with the raw query coordinate; the FiLM
decoder lifts the coordinate through fixed random Fourier features and modulates
two FiLM blocks with `z_geom`. The parameter counts are fixed fingerprints of the
two architectures.

In [ ]:
_tok  = torch.zeros(2, 30, 5)
_pmsk = torch.zeros(2, 30, dtype=torch.bool)
_qxy  = torch.rand(2, K_PTS, 2)
for name, M in [("WP2-base", PooledTransformer), ("WP2-FiLM", PooledTransformerFiLM)]:
    _m = M()
    _out = _m(_tok, _pmsk, _qxy)
    _z = _m.encode(_tok, _pmsk)
    n_p = sum(p.numel() for p in _m.parameters())
    print(f"{name:<10} pred {tuple(_out.shape)}  z_geom {tuple(_z.shape)}  params {n_p:,}")
del _tok, _pmsk, _qxy

## 4. Training

`point_mse` is plain MSE over the sampled fluid query points (no mask needed:
`TokenDataset` samples only fluid cells). `train_model` is a single reusable loop
(early stopping + best-checkpoint) returning the best weights and the per-epoch
history so the two runs can be overlaid.

In [ ]:
def point_mse(pred, target):
    """MSE over sampled fluid query points (all points are pre-filtered to fluid)."""
    return ((pred - target) ** 2).mean()


def train_model(model, train_loader, val_loader, ckpt_path, tag="model",
                num_epochs=NUM_EPOCHS, lr=LR, patience=PATIENCE):
    """Train one model. Returns (best_weights, history) where history is a list of
    (epoch, train_loss, val_loss)."""
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_loss, best_weights, patience_counter, history = float("inf"), None, 0, []

    for epoch in range(num_epochs):
        # --- train ---
        model.train()
        train_loss = 0.0
        for tok, pmsk, qxy, tgt in tqdm(train_loader, desc=f"[{tag}] epoch {epoch + 1}"):
            tok, pmsk, qxy, tgt = (tok.to(DEVICE), pmsk.to(DEVICE), qxy.to(DEVICE), tgt.to(DEVICE))
            optimizer.zero_grad()
            loss = point_mse(model(tok, pmsk, qxy), tgt)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # --- validate ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for tok, pmsk, qxy, tgt in val_loader:
                tok, pmsk, qxy, tgt = (tok.to(DEVICE), pmsk.to(DEVICE), qxy.to(DEVICE), tgt.to(DEVICE))
                val_loss += point_mse(model(tok, pmsk, qxy), tgt).item()
        val_loss /= len(val_loader)
        history.append((epoch + 1, train_loss, val_loss))
        print(f"[{tag}] epoch {epoch + 1}/{num_epochs} | train {train_loss:.4f} | val {val_loss:.4f}")

        # --- checkpoint + early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, ckpt_path)
            patience_counter = 0
            print(f"  -> new best ({val_loss:.4f}); checkpoint saved")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  -> early stop at epoch {epoch + 1}")
                break
    return best_weights, history

In [ ]:
model_base = PooledTransformer()
best_weights_base, history_base = train_model(model_base, train_loader, val_loader,
                                              "wp2_base_best.pt", tag="base")

model_film = PooledTransformerFiLM()
best_weights_film, history_film = train_model(model_film, train_loader, val_loader,
                                              "wp2_film_best.pt", tag="FiLM")

## 5. Evaluation

`predict_full_map` reconstructs the full 78x78 field for one case through the
shared `decode(z, q)` interface, so it serves both models unchanged.
`evaluate_model` loops it over the test set; metrics come from
`urbanformer.metrics.field_metrics`, identical to every other WP.

In [ ]:
def predict_full_map(model, case_dir, device, chunk=1024):
    """Reconstruct the full Ny x Nx velocity map for one case (all cells decoded)."""
    tokens = np.load(case_dir / "building_tokens.npy").astype(np.float32)
    U_mid  = np.load(case_dir / "U_mid.npy").astype(np.float32)
    fluid  = np.load(case_dir / "fluid_mask_mid.npy").astype(np.float32)
    Ny, Nx = U_mid.shape

    x_ch = (np.arange(Nx, dtype=np.float32) + 0.5) / Nx
    y_ch = (np.arange(Ny, dtype=np.float32) + 0.5) / Ny
    yy, xx = np.meshgrid(y_ch, x_ch, indexing="ij")
    query_all = torch.from_numpy(np.stack([xx.flatten(), yy.flatten()], axis=-1))  # (Ny*Nx, 2)

    tok_t  = torch.from_numpy(tokens).unsqueeze(0).to(device)                        # (1, N_b, 5)
    pmsk_t = torch.zeros(1, tokens.shape[0], dtype=torch.bool, device=device)
    z = model.encode(tok_t, pmsk_t)                                                 # (1, d_model)

    preds = []
    for start in range(0, query_all.shape[0], chunk):
        q = query_all[start:start + chunk].unsqueeze(0).to(device)                  # (1, c, 2)
        preds.append(model.decode(z, q).squeeze(0).cpu())                           # (c,)
    pred_map = torch.cat(preds).reshape(Ny, Nx).numpy()
    return pred_map, U_mid, fluid


def evaluate_model(model, best_weights, test_dirs, device=DEVICE):
    """Run predict_full_map over the test set; return stacked arrays + field metrics."""
    model.load_state_dict(best_weights)
    model = model.to(device).eval()
    all_preds, all_targets, all_masks = [], [], []
    with torch.no_grad():
        for case_dir in tqdm(test_dirs, desc="evaluating"):
            pred_map, U_mid, fluid = predict_full_map(model, case_dir, device)
            all_preds.append(pred_map); all_targets.append(U_mid); all_masks.append(fluid)
    P = torch.from_numpy(np.stack(all_preds))
    T = torch.from_numpy(np.stack(all_targets))
    M = torch.from_numpy(np.stack(all_masks))
    return P, T, M, field_metrics(P, T, M)

In [ ]:
P_base, T_base, M_base, metrics_base = evaluate_model(model_base, best_weights_base, test_dirs)
print(f"WP2-base | RMSE {metrics_base['RMSE']:.4f} | MAE {metrics_base['MAE']:.4f} | R2 {metrics_base['R2']:.4f}")

P_film, T_film, M_film, metrics_film = evaluate_model(model_film, best_weights_film, test_dirs)
print(f"WP2-FiLM | RMSE {metrics_film['RMSE']:.4f} | MAE {metrics_film['MAE']:.4f} | R2 {metrics_film['R2']:.4f}")

In [ ]:
# Comparison table: the WP2 story in one place.
rows = [
    ("WP1 - U-Net (raster)",         WP1_RMSE,             WP1_MAE,             WP1_R2),
    ("WP2 - Pooled (base)",          metrics_base["RMSE"], metrics_base["MAE"], metrics_base["R2"]),
    ("WP2 - Pooled + Fourier/FiLM",  metrics_film["RMSE"], metrics_film["MAE"], metrics_film["R2"]),
]
print(f"{'Model':<32}{'RMSE':>9}{'MAE':>9}{'R2':>9}")
for name, r, m, r2 in rows:
    print(f"{name:<32}{r:>9.4f}{m:>9.4f}{r2:>9.4f}")

## 6. Visualization

Ground truth, prediction, and absolute error for representative cases, plus the
training curves. Flat curves near the target variance are the signature of the
mean-field collapse the base model falls into.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

case_rmse_film = per_case_rmse(P_film, T_film, M_film)
best_idx  = int(np.nanargmin(case_rmse_film))
worst_idx = int(np.nanargmax(case_rmse_film))


def plot_simulation_u(ax, u_norm, solid_mask, title="", vmin=-1.0, vmax=3.0):
    colors = [(0, 0, 0.5), (0, 0.8, 0.8), (0, 0.25, 0), (0.7, 0.7, 0), (0.4, 0, 0)]
    positions = [0, 0.15, (0 - vmin) / (vmax - vmin), 0.55, 1]
    cmap = mcolors.LinearSegmentedColormap.from_list("custom", list(zip(positions, colors)))
    data = u_norm.copy(); data[solid_mask] = 0
    cax = ax.imshow(data, cmap=cmap, interpolation="bicubic", vmin=vmin, vmax=vmax, origin="upper")
    ax.imshow(np.where(solid_mask, 0, np.nan), cmap="gray", interpolation="nearest", origin="upper", zorder=2)
    ax.set_aspect("equal"); ax.set_title(title); ax.grid(False)
    plt.colorbar(cax, ax=ax, orientation="horizontal", fraction=0.046, pad=0.15, extend="both")


def plot_case(idx, title):
    pred, target, mask = P_film[idx].numpy(), T_film[idx].numpy(), M_film[idx].numpy()
    solid = mask == 0
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{title}  |  FiLM RMSE={case_rmse_film[idx]:.4f}", fontsize=13)
    plot_simulation_u(axes[0], target, solid, title="CFD (ground truth)")
    plot_simulation_u(axes[1], pred,   solid, title="WP2-FiLM prediction")
    im = axes[2].imshow(np.abs(pred - target) * mask, cmap="hot", origin="upper")
    axes[2].set_title("|error|"); plt.colorbar(im, ax=axes[2], orientation="horizontal", fraction=0.046, pad=0.15)
    plt.tight_layout(); plt.show()

plot_case(best_idx,  "Best case (FiLM)")
plot_case(worst_idx, "Worst case (FiLM)")

In [ ]:
# Training curves: base vs FiLM.
fig, ax = plt.subplots(figsize=(7, 4))
for hist, tag in [(history_base, "base"), (history_film, "FiLM")]:
    ep = [h[0] for h in hist]
    ax.plot(ep, [h[1] for h in hist], "--", label=f"{tag} train")
    ax.plot(ep, [h[2] for h in hist], "-",  label=f"{tag} val")
ax.set_xlabel("epoch"); ax.set_ylabel("point MSE"); ax.legend()
ax.set_title("WP2 training curves (flat near target variance = mean field)")
plt.tight_layout(); plt.show()